In [1]:
import time
import json
import statistics
from transformers import pipeline

from dataset import CANDIDATE_LABELS, TEXTS, TRUE_LABELS

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("Candidate labels:", CANDIDATE_LABELS)
print("Total examples:", len(TEXTS))

from collections import Counter
print("Examples per class:", Counter(TRUE_LABELS))

print("\nSample examples:")
for text, label in list(zip(TEXTS, TRUE_LABELS))[:3]:
    print(f"  [{label}] {text}")


Candidate labels: ['world news', 'sports', 'business', 'science and technology']
Total examples: 40
Examples per class: Counter({'world news': 10, 'sports': 10, 'business': 10, 'science and technology': 10})

Sample examples:
  [world news] The United Nations held an emergency session to discuss the ongoing border dispute between the two nations.
  [world news] Protesters gathered outside the parliament building demanding political reform.
  [world news] The prime minister announced a new diplomatic agreement with neighboring countries.


In [8]:
model ="facebook/bart-large-mnli"

In [10]:
# Loading the model
print("Loading facebook/bart-large-mnli ...")
classifier = pipeline(
    "zero-shot-classification",
    model=model,
    device=-1,  # force CPU
)
print("Loaded.")


Loading facebook/bart-large-mnli ...


Device set to use cpu


Loaded.


In [11]:
#testing the output
example_text = TEXTS[0]
example_true_label = TRUE_LABELS[0]

result = classifier(example_text, CANDIDATE_LABELS)

In [12]:
print("Input text:", example_text)
print("True label:", example_true_label)
print("result is here")
print(result)


for label, score in zip(result["labels"], result["scores"]):
    print(f"  {label:30s} -> {score:.4f}")

Input text: The United Nations held an emergency session to discuss the ongoing border dispute between the two nations.
True label: world news
result is here
{'sequence': 'The United Nations held an emergency session to discuss the ongoing border dispute between the two nations.', 'labels': ['world news', 'business', 'science and technology', 'sports'], 'scores': [0.9140632152557373, 0.029660455882549286, 0.028874559327960014, 0.027401825413107872]}
  world news                     -> 0.9141
  business                       -> 0.0297
  science and technology         -> 0.0289
  sports                         -> 0.0274


In [13]:
def get_model_size_mb(model):
    """In-memory size of model parameters, based on dtype/element size."""
    total_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    total_bytes += sum(b.numel() * b.element_size() for b in model.buffers())
    return total_bytes / (1024 ** 2)

In [15]:
def run_benchmark(classifier, texts, true_labels, label_name):
    predictions = []
    latencies = []

    for text in texts:
        start = time.perf_counter()
        result = classifier(text, CANDIDATE_LABELS)
        end = time.perf_counter()

        latencies.append(end - start)
        predictions.append(result["labels"][0])  # top predicted label

    correct = sum(p == t for p, t in zip(predictions, true_labels))
    accuracy = correct / len(true_labels)

    stats = {
        "run_name": label_name,
        "accuracy": accuracy,
        "num_examples": len(texts),
        "total_time_sec": sum(latencies),
        "mean_latency_sec": statistics.mean(latencies),
        "median_latency_sec": statistics.median(latencies),
        "p95_latency_sec": statistics.quantiles(latencies, n=20)[18],  # ~95th percentile
        "throughput_examples_per_sec": len(texts) / sum(latencies),
        "predictions": predictions,
    }
    return stats

In [16]:
model_size_mb = get_model_size_mb(classifier.model)
print(f"Model in-memory size (FP32 params): {model_size_mb:.2f} MB")

Model in-memory size (FP32 params): 1553.89 MB


In [17]:
# run the benchmark 
results = run_benchmark(classifier, TEXTS, TRUE_LABELS, "baseline_fp32")

In [18]:
results["model_size_mb"] = model_size_mb

In [19]:
print("===== BASELINE RESULTS =====")
print(f"Accuracy:            {results['accuracy']*100:.1f}%")
print(f"Mean latency:        {results['mean_latency_sec']*1000:.1f} ms")
print(f"Median latency:      {results['median_latency_sec']*1000:.1f} ms")
print(f"P95 latency:         {results['p95_latency_sec']*1000:.1f} ms")
print(f"Throughput:          {results['throughput_examples_per_sec']:.2f} examples/sec")
print(f"Model size (FP32):   {results['model_size_mb']:.2f} MB")


===== BASELINE RESULTS =====
Accuracy:            85.0%
Mean latency:        681.8 ms
Median latency:      676.7 ms
P95 latency:         826.0 ms
Throughput:          1.47 examples/sec
Model size (FP32):   1553.89 MB


In [20]:
# we are seeing where it went wrong 
from collections import defaultdict

per_class_correct = defaultdict(int)
per_class_total = defaultdict(int)
mistakes = []

for text, true_label, pred_label in zip(TEXTS, TRUE_LABELS, results["predictions"]):
    per_class_total[true_label] += 1
    if true_label == pred_label:
        per_class_correct[true_label] += 1
    else:
        mistakes.append((text, true_label, pred_label))

print("Per-class accuracy:")
for label in CANDIDATE_LABELS:
    total = per_class_total[label]
    correct = per_class_correct[label]
    print(f"  {label:25s} {correct}/{total} = {correct/total*100:.1f}%")

print(f"\nTotal mistakes: {len(mistakes)}")
for text, true_label, pred_label in mistakes:
    print(f"  TRUE: {true_label:25s} PRED: {pred_label:25s} TEXT: {text}")


Per-class accuracy:
  world news                9/10 = 90.0%
  sports                    10/10 = 100.0%
  business                  6/10 = 60.0%
  science and technology    9/10 = 90.0%

Total mistakes: 6
  TRUE: world news                PRED: business                  TEXT: A new trade deal was signed between the two countries after years of negotiation.
  TRUE: business                  PRED: world news                TEXT: The central bank raised interest rates to curb inflation.
  TRUE: business                  PRED: science and technology    TEXT: Shares of the tech giant hit an all time high this week.
  TRUE: business                  PRED: world news                TEXT: Unemployment figures dropped to their lowest level in a decade.
  TRUE: business                  PRED: world news                TEXT: Consumer spending rose sharply during the holiday shopping season.
  TRUE: science and technology    PRED: world news                TEXT: The study published in a leading jo

In [21]:
with open("baseline_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved to baseline_results.json")

Saved to baseline_results.json
